<a href="https://colab.research.google.com/github/davidsthename/TrashNet-Image-Classification/blob/main/Trashnet_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/TrashNet_Project/dataset-resized"


In [ ]:
# --------- DATA LOADING ---------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [ ]:
dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# --------- CUSTOM CNN ---------
class TrashNetCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.pool = nn.MaxPool2d(2,2)
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(128*28*28, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128*28*28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
num_classes = len(dataset.classes)
model_cnn = TrashNetCNN(num_classes=num_classes).to(device)

In [ ]:
# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=0.001)

In [ ]:
# --------- TRAIN CUSTOM CNN ---------
num_epochs = 5  # Adjust for faster testing, increase later
for epoch in range(num_epochs):
    model_cnn.train()
    running_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*images.size(0)
        _, predicted = torch.max(outputs,1)
        correct += (predicted==labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss/total
    train_acc = correct/total

In [ ]:
 # Validation
model_cnn.eval()
val_loss, val_correct, val_total = 0,0,0
with torch.no_grad():
  for images, labels in val_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model_cnn(images)
    loss = criterion(outputs, labels)
    val_loss += loss.item()*images.size(0)
    _, predicted = torch.max(outputs,1)
    val_correct += (predicted==labels).sum().item()
    val_total += labels.size(0)
val_loss /= val_total
val_acc = val_correct/val_total
print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}")

Epoch 5/5 | Train Acc: 0.753 | Val Acc: 0.698


In [ ]:
# --------- TRANSFER LEARNING (ResNet50) ---------
resnet = models.resnet50(pretrained=True)
for param in resnet.parameters():
    param.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

optimizer_resnet = torch.optim.Adam(resnet.fc.parameters(), lr=0.001)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s]


In [ ]:
# Train ResNet50 (small epochs for testing)
num_epochs_resnet = 3
for epoch in range(num_epochs_resnet):
    resnet.train()
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_resnet.step()
        _, predicted = torch.max(outputs,1)
        correct += (predicted==labels).sum().item()
        total += labels.size(0)
    train_acc = correct/total
    print(f"ResNet Epoch {epoch+1}/{num_epochs_resnet} | Train Acc: {train_acc:.3f}")

ResNet Epoch 1/3 | Train Acc: 0.612
ResNet Epoch 2/3 | Train Acc: 0.801
ResNet Epoch 3/3 | Train Acc: 0.824


In [ ]:
# --------- TEST ON REAL IMAGE ---------
from google.colab import files
uploaded = files.upload()  # Upload any image

img_path = list(uploaded.keys())[0]
image = Image.open(img_path).convert("RGB")
image_tensor = transform(image).unsqueeze(0).to(device)

Saving Untitled.png to Untitled (2).png


In [ ]:
# Custom CNN prediction
model_cnn.eval()
with torch.no_grad():
    output = model_cnn(image_tensor)
    _, pred = torch.max(output,1)
print("Custom CNN predicts:", dataset.classes[pred.item()])

Custom CNN predicts: cardboard


In [ ]:
# ResNet prediction
resnet.eval()
with torch.no_grad():
    output = resnet(image_tensor)
    _, pred = torch.max(output,1)
print("ResNet50 predicts:", dataset.classes[pred.item()])

ResNet50 predicts: cardboard
